# CC - Saltim: Pipeline de Dados, EDA e Baseline Preditivo

Notebook de implementação da entrega da CC com foco em:
- carregamento dos dados,
- análise exploratória inicial,
- limpeza e transformação,
- tratamento de outliers,
- baseline de previsão semanal de pedidos.

A estrutura segue princípios de reprodutibilidade, modularização e documentação acadêmica.

## Parte 1 — Carga, EDA e Tratamento dos Dados

Entrega da **Parte 1** do roteiro: carregamento dos dados, análise exploratória inicial e primeiras etapas de tratamento e transformação (limpeza, normalização, encoding, eliminação de outliers).

Estrutura modularizada e reprodutível, distribuída pelas seções de apoio abaixo:
- **Seção 1** — set up de ambiente e dependências
- **Seção 2** — configuração centralizada (caminhos, parâmetros, hiperparâmetros)
- **Seção 3** — estruturas de dados tipadas (`QualitySummary`, `PipelineState`)
- **Seção 4** — funções modulares (`load_tables`, `normalize_tables`, `create_daily_frame`, `remove_outliers_iqr_with_exceptions`, `build_weekly_dataset`)
- **Seção 5** — execução do pipeline (relatórios de qualidade, base diária, base semanal)
- **Seção 7** — visualizações avançadas de EDA (correlação, séries longas, heatmap sazonal)

## 1. Configuração do Ambiente e Dependências
Instalação/importação de dependências, verificação de versões e inicialização de configurações reprodutíveis.

In [ ]:
from __future__ import annotations

import random
import warnings
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

sns.set_theme(style="whitegrid")

print("Ambiente pronto")
print(f"pandas: {pd.__version__}")
print(f"numpy: {np.__version__}")

: 

## 2. Configuração e Constantes
Bloco central de configuração com caminhos, parâmetros de tratamento e hiperparâmetros dos baselines.

In [ ]:
CONFIG: dict[str, Any] = {
    "project_root": Path.cwd(),
    "data_dir": Path("data"),
    "test_weeks": 16,
    "iqr_multiplier": 1.5,
    "event_exception_cols": ["is_holiday", "is_carnaval", "is_sao_joao", "is_summer"],
    "expected_files": {
        "estoque": "estoque_entradas_sintetico_2023_2026_mar.xlsx",
        "fichas": "fichas_tecnicas_ids.xlsx",
        "ingredientes": "lista_ingredientes.xlsx",
        "receitas": "lista_receitas.xlsx",
        "pedidos": "pedidos_sinteticos_2023_2026_mar.xlsx",
        "resumo_diario": "resumo_diario_pedidos_2023_2026_mar.xlsx",
        "resumo_mensal": "resumo_mensal_pedidos_2023_2026_mar.xlsx",
    },
}

print("Diretório do projeto:", CONFIG["project_root"])
print("Diretório de dados:", CONFIG["data_dir"].resolve())

## 3. Estruturas de Dados
Definição de dataclasses e estruturas tipadas para transportar dados e resultados de forma clara.

In [ ]:
@dataclass
class QualitySummary:
    tabela: str
    linhas: int
    colunas: int
    nulos_total: int
    duplicatas: int


@dataclass
class ModelArtifacts:
    rmse: dict[str, float]
    test_frame: pd.DataFrame
    model: Pipeline


@dataclass
class PipelineState:
    tabelas_raw: dict[str, pd.DataFrame]
    tabelas_clean: dict[str, pd.DataFrame]
    diario: pd.DataFrame
    diario_sem_outliers: pd.DataFrame
    semanal: pd.DataFrame
    outlier_audit: pd.DataFrame
    model_artifacts: ModelArtifacts | None = None

## 4. Funções de Processamento
Funções modulares para carregamento, limpeza, transformação, detecção de outliers e modelagem baseline.

In [ ]:
def to_snake_case(columns: list[str]) -> list[str]:
    return [
        c.strip()
        .replace(" ", "_")
        .replace("-", "_")
        .replace("/", "_")
        .replace("(", "")
        .replace(")", "")
        .lower()
        for c in columns
    ]


def load_tables(config: dict[str, Any]) -> dict[str, pd.DataFrame]:
    tables: dict[str, pd.DataFrame] = {}
    for key, file_name in config["expected_files"].items():
        path = config["data_dir"] / file_name
        if not path.exists():
            raise FileNotFoundError(f"Arquivo não encontrado: {path}")
        df = pd.read_excel(path)
        df.columns = to_snake_case(df.columns.tolist())
        tables[key] = df
    return tables


def build_quality_report(tables: dict[str, pd.DataFrame]) -> pd.DataFrame:
    records: list[QualitySummary] = []
    for name, df in tables.items():
        records.append(
            QualitySummary(
                tabela=name,
                linhas=int(df.shape[0]),
                colunas=int(df.shape[1]),
                nulos_total=int(df.isna().sum().sum()),
                duplicatas=int(df.duplicated().sum()),
            )
        )
    return pd.DataFrame([r.__dict__ for r in records]).sort_values("tabela")


def normalize_tables(tables: dict[str, pd.DataFrame]) -> dict[str, pd.DataFrame]:
    clean = {k: v.copy() for k, v in tables.items()}

    clean["pedidos"]["data_hora"] = pd.to_datetime(clean["pedidos"]["data_hora"], errors="coerce")
    clean["pedidos"] = clean["pedidos"].rename(columns={"id_produto": "id_receita"})
    clean["pedidos"]["id_receita"] = clean["pedidos"]["id_receita"].astype("Int64")
    clean["pedidos"]["quantidade"] = pd.to_numeric(clean["pedidos"]["quantidade"], errors="coerce")

    clean["resumo_diario"]["date"] = pd.to_datetime(clean["resumo_diario"]["date"], errors="coerce")
    clean["estoque"]["data_recebimento"] = pd.to_datetime(clean["estoque"]["data_recebimento"], errors="coerce")
    clean["estoque"]["data_validade"] = pd.to_datetime(clean["estoque"]["data_validade"], errors="coerce")

    clean["fichas"] = clean["fichas"].dropna(subset=["qtd_ingrediente", "custo_ingrediente"])

    for table_name, text_cols in {
        "ingredientes": ["ingrediente", "un_ingrediente"],
        "receitas": ["receita", "tipo"],
    }.items():
        for col in text_cols:
            clean[table_name][col] = clean[table_name][col].astype(str).str.strip().str.upper()

    for name, df in clean.items():
        clean[name] = df.drop_duplicates().reset_index(drop=True)

    clean["pedidos"] = clean["pedidos"].dropna(subset=["data_hora", "id_receita", "quantidade"])
    clean["pedidos"] = clean["pedidos"][clean["pedidos"]["quantidade"] > 0]

    return clean


def create_daily_frame(tables: dict[str, pd.DataFrame]) -> pd.DataFrame:
    pedidos = tables["pedidos"].copy()
    resumo_diario = tables["resumo_diario"].copy().rename(columns={"pedidos_dia": "pedidos_dia_resumo"})

    pedidos["date"] = pedidos["data_hora"].dt.floor("D")

    daily = (
        pedidos.groupby("date", as_index=False)
        .agg(
            pedidos_dia=("id_pedido", "nunique"),
            itens_dia=("quantidade", "sum"),
            linhas_itens=("id_pedido", "size"),
        )
        .sort_values("date")
    )

    event_cols = ["date", "pedidos_dia_resumo", "is_holiday", "is_carnaval", "is_sao_joao", "is_summer"]
    daily = daily.merge(resumo_diario[event_cols], on="date", how="left")

    for col in ["is_holiday", "is_carnaval", "is_sao_joao", "is_summer"]:
        daily[col] = daily[col].fillna(0).astype(int)

    daily["diff_resumo"] = daily["pedidos_dia"] - daily["pedidos_dia_resumo"].fillna(0)
    return daily


def remove_outliers_iqr_with_exceptions(
    df: pd.DataFrame,
    target_col: str,
    event_cols: list[str],
    multiplier: float = 1.5,
) -> tuple[pd.DataFrame, pd.DataFrame, dict[str, float]]:
    work = df.copy()
    q1 = work[target_col].quantile(0.25)
    q3 = work[target_col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - multiplier * iqr
    upper = q3 + multiplier * iqr

    outlier_mask = (work[target_col] < lower) | (work[target_col] > upper)
    event_mask = work[event_cols].sum(axis=1) > 0
    weekend_mask = work["date"].dt.weekday >= 5

    # Exceções temporais: mantém picos de eventos e fins de semana.
    removable_mask = outlier_mask & ~(event_mask | weekend_mask)

    audit = work.loc[outlier_mask, ["date", target_col, *event_cols]].copy()
    audit["removido"] = removable_mask[outlier_mask].values

    cleaned = work.loc[~removable_mask].copy().reset_index(drop=True)
    bounds = {"q1": float(q1), "q3": float(q3), "lower": float(lower), "upper": float(upper)}

    return cleaned, audit, bounds


def build_weekly_dataset(daily_clean: pd.DataFrame) -> pd.DataFrame:
    weekly = (
        daily_clean.set_index("date")
        .resample("W-MON", label="left", closed="left")
        .agg(
            pedidos_semana=("pedidos_dia", "sum"),
            itens_semana=("itens_dia", "sum"),
            feriado_semana=("is_holiday", "max"),
            carnaval_semana=("is_carnaval", "max"),
            sao_joao_semana=("is_sao_joao", "max"),
            verao_semana=("is_summer", "max"),
        )
        .reset_index()
        .rename(columns={"date": "semana_inicio"})
    )

    weekly["ano"] = weekly["semana_inicio"].dt.year
    weekly["mes"] = weekly["semana_inicio"].dt.month.astype(str)
    weekly["semana_ano"] = weekly["semana_inicio"].dt.isocalendar().week.astype(int)
    weekly["trend"] = np.arange(len(weekly), dtype=float)
    weekly["lag_1"] = weekly["pedidos_semana"].shift(1)
    weekly["rolling_4"] = weekly["pedidos_semana"].shift(1).rolling(4).mean()

    weekly = weekly.dropna().reset_index(drop=True)
    return weekly


def evaluate_baselines(weekly_df: pd.DataFrame, test_weeks: int = 16) -> ModelArtifacts:
    if len(weekly_df) <= test_weeks + 8:
        raise ValueError("Série semanal curta demais para separar treino/teste com segurança.")

    split_idx = len(weekly_df) - test_weeks
    train = weekly_df.iloc[:split_idx].copy()
    test = weekly_df.iloc[split_idx:].copy()

    y_train = train["pedidos_semana"]
    y_test = test["pedidos_semana"]

    naive_pred = weekly_df["pedidos_semana"].shift(1).iloc[split_idx:]
    rmse_naive = float(mean_squared_error(y_test, naive_pred, squared=False))

    num_features = [
        "itens_semana",
        "feriado_semana",
        "carnaval_semana",
        "sao_joao_semana",
        "verao_semana",
        "semana_ano",
        "trend",
        "lag_1",
        "rolling_4",
    ]
    cat_features = ["mes"]

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", StandardScaler(), num_features),
            ("cat", OneHotEncoder(handle_unknown="ignore"), cat_features),
        ]
    )

    model = Pipeline(
        steps=[
            ("prep", preprocessor),
            ("reg", LinearRegression()),
        ]
    )

    X_train = train[num_features + cat_features]
    X_test = test[num_features + cat_features]

    model.fit(X_train, y_train)
    reg_pred = model.predict(X_test)
    rmse_reg = float(mean_squared_error(y_test, reg_pred, squared=False))

    test_frame = test[["semana_inicio", "pedidos_semana"]].copy()
    test_frame["pred_naive"] = naive_pred.values
    test_frame["pred_regressao"] = reg_pred

    return ModelArtifacts(
        rmse={"naive": rmse_naive, "regressao": rmse_reg},
        test_frame=test_frame,
        model=model,
    )

In [ ]:
# Correções de robustez: remover semanas parciais e compatibilizar RMSE.
def build_weekly_dataset(daily_clean: pd.DataFrame) -> pd.DataFrame:
    weekly = (
        daily_clean.set_index("date")
        .resample("W-MON", label="left", closed="left")
        .agg(
            pedidos_semana=("pedidos_dia", "sum"),
            itens_semana=("itens_dia", "sum"),
            feriado_semana=("is_holiday", "max"),
            carnaval_semana=("is_carnaval", "max"),
            sao_joao_semana=("is_sao_joao", "max"),
            verao_semana=("is_summer", "max"),
            dias_observados=("pedidos_dia", "size"),
        )
        .reset_index()
        .rename(columns={"date": "semana_inicio"})
    )

    weekly = weekly[weekly["dias_observados"] == 7].copy()
    weekly = weekly.drop(columns=["dias_observados"])

    weekly["ano"] = weekly["semana_inicio"].dt.year
    weekly["mes"] = weekly["semana_inicio"].dt.month.astype(str)
    weekly["semana_ano"] = weekly["semana_inicio"].dt.isocalendar().week.astype(int)
    weekly["trend"] = np.arange(len(weekly), dtype=float)
    weekly["lag_1"] = weekly["pedidos_semana"].shift(1)
    weekly["rolling_4"] = weekly["pedidos_semana"].shift(1).rolling(4).mean()

    weekly = weekly.dropna().reset_index(drop=True)
    return weekly


def evaluate_baselines(weekly_df: pd.DataFrame, test_weeks: int = 16) -> ModelArtifacts:
    if len(weekly_df) <= test_weeks + 8:
        raise ValueError("Série semanal curta demais para separar treino/teste com segurança.")

    split_idx = len(weekly_df) - test_weeks
    train = weekly_df.iloc[:split_idx].copy()
    test = weekly_df.iloc[split_idx:].copy()

    y_train = train["pedidos_semana"]
    y_test = test["pedidos_semana"]

    naive_pred = weekly_df["pedidos_semana"].shift(1).iloc[split_idx:]
    rmse_naive = float(np.sqrt(mean_squared_error(y_test, naive_pred)))

    num_features = [
        "itens_semana",
        "feriado_semana",
        "carnaval_semana",
        "sao_joao_semana",
        "verao_semana",
        "semana_ano",
        "trend",
        "lag_1",
        "rolling_4",
    ]
    cat_features = ["mes"]

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", StandardScaler(), num_features),
            ("cat", OneHotEncoder(handle_unknown="ignore"), cat_features),
        ]
    )

    model = Pipeline(
        steps=[
            ("prep", preprocessor),
            ("reg", LinearRegression()),
        ]
    )

    X_train = train[num_features + cat_features]
    X_test = test[num_features + cat_features]

    model.fit(X_train, y_train)
    reg_pred = model.predict(X_test)
    rmse_reg = float(np.sqrt(mean_squared_error(y_test, reg_pred)))

    test_frame = test[["semana_inicio", "pedidos_semana"]].copy()
    test_frame["pred_naive"] = naive_pred.values
    test_frame["pred_regressao"] = reg_pred

    return ModelArtifacts(
        rmse={"naive": rmse_naive, "regressao": rmse_reg},
        test_frame=test_frame,
        model=model,
    )

## 5. Execução do Pipeline
Execução completa do pipeline, com outputs intermediários e visualizações iniciais.

In [ ]:
# 1) Carga e diagnóstico inicial
tabelas_raw = load_tables(CONFIG)
quality_raw = build_quality_report(tabelas_raw)
print("Relatório de qualidade (dados brutos):")
display(quality_raw)

# 2) Limpeza e normalização
tabelas_clean = normalize_tables(tabelas_raw)
quality_clean = build_quality_report(tabelas_clean)
print("Relatório de qualidade (dados tratados):")
display(quality_clean)

# 3) Base diária
diario = create_daily_frame(tabelas_clean)
print("Amostra da base diária:")
display(diario.head())

# 4) Outliers IQR + exceção temporal
diario_sem_outliers, outlier_audit, bounds = remove_outliers_iqr_with_exceptions(
    diario,
    target_col="pedidos_dia",
    event_cols=CONFIG["event_exception_cols"],
    multiplier=CONFIG["iqr_multiplier"],
)

print("Resumo dos limites de outlier:", bounds)
print("Outliers detectados:", int(outlier_audit.shape[0]))
print("Outliers removidos:", int(outlier_audit["removido"].sum()))

# 5) Base semanal + baseline
semanal = build_weekly_dataset(diario_sem_outliers)
artifacts = evaluate_baselines(semanal, test_weeks=CONFIG["test_weeks"])

print("RMSE dos baselines:")
for name, score in artifacts.rmse.items():
    print(f"- {name}: {score:.3f}")

# 6) Visualizações
fig, axes = plt.subplots(2, 1, figsize=(14, 10), constrained_layout=True)

axes[0].plot(diario["date"], diario["pedidos_dia"], label="Série diária bruta", alpha=0.6)
axes[0].plot(diario_sem_outliers["date"], diario_sem_outliers["pedidos_dia"], label="Série diária tratada", alpha=0.9)
axes[0].set_title("Pedidos diários - antes e depois do tratamento de outliers")
axes[0].set_xlabel("Data")
axes[0].set_ylabel("Pedidos")
axes[0].legend()

axes[1].plot(artifacts.test_frame["semana_inicio"], artifacts.test_frame["pedidos_semana"], marker="o", label="Real")
axes[1].plot(artifacts.test_frame["semana_inicio"], artifacts.test_frame["pred_naive"], marker="o", label="Naive")
axes[1].plot(artifacts.test_frame["semana_inicio"], artifacts.test_frame["pred_regressao"], marker="o", label="Regressão")
axes[1].set_title("Teste temporal - previsão semanal")
axes[1].set_xlabel("Semana")
axes[1].set_ylabel("Pedidos por semana")
axes[1].legend()

plt.show()

pipeline_state = PipelineState(
    tabelas_raw=tabelas_raw,
    tabelas_clean=tabelas_clean,
    diario=diario,
    diario_sem_outliers=diario_sem_outliers,
    semanal=semanal,
    outlier_audit=outlier_audit,
    model_artifacts=artifacts,
)

print("Pipeline executado com sucesso.")

## Parte 2 — Modelo Inicial Simples (Baseline)

Entrega da **Parte 2** do roteiro: treinamento de um modelo inicial simples adequado ao problema de regressão (previsão semanal de pedidos), com aplicação de métricas de desempenho sobre um conjunto de validação temporal.

Implementação na função `evaluate_baselines` (Seção 4), executada acima como parte do fluxo da Seção 5:
- **Modelo simples**: regressão linear (`LinearRegression`) com `StandardScaler` para features numéricas e `OneHotEncoder` para o mês — encapsulados em um `Pipeline` do scikit-learn.
- **Comparação interna**: baseline naive (lag-1) como referência mínima.
- **Validação**: split temporal — últimas `test_weeks` (16) semanas reservadas como conjunto de teste.
- **Métrica**: RMSE para baseline naive e regressão (impressos acima como "RMSE dos baselines"), com gráfico real × naive × regressão na figura inferior.

Os artefatos do baseline ficam armazenados em `pipeline_state.model_artifacts` e servem como referência para as Partes 3 (comparação com modelos mais complexos) e 4 (ajuste de hiperparâmetros).

## 6. Validações e Testes
Validações rápidas para garantir consistência de dados e comportamento esperado do pipeline.

In [ ]:
# Testes de sanidade de dados
assert len(pipeline_state.tabelas_raw) == 7, "Quantidade de tabelas carregadas diferente do esperado."
assert all(df.shape[0] > 0 for df in pipeline_state.tabelas_raw.values()), "Existe tabela vazia nos dados brutos."
assert pipeline_state.diario_sem_outliers["pedidos_dia"].min() > 0, "Pedidos diários tratados contém valores inválidos."
assert pipeline_state.semanal["pedidos_semana"].isna().sum() == 0, "Target semanal possui nulos após preparação."

# Testes de consistência de modelagem
assert {"naive", "regressao"}.issubset(set(pipeline_state.model_artifacts.rmse.keys())), "Métricas RMSE incompletas."
assert all(v >= 0 for v in pipeline_state.model_artifacts.rmse.values()), "RMSE inválido (negativo)."

# Teste da regra de exceção temporal para outliers
if not pipeline_state.outlier_audit.empty:
    removed_events = pipeline_state.outlier_audit.loc[
        pipeline_state.outlier_audit["removido"],
        ["is_holiday", "is_carnaval", "is_sao_joao", "is_summer"],
    ].sum(axis=1)
    assert (removed_events == 0).all(), "Foram removidos outliers em dias de evento, contrariando a regra."

print("Todos os testes básicos foram aprovados.")

## 7. Visualizações Avançadas: Correlação e Séries Temporais Longas
Nesta seção expandimos a análise visual com foco em:
- matriz de correlação entre variáveis numéricas,
- séries temporais completas (2023-2026),
- tendências de curto e longo prazo,
- sazonalidade semanal por ano.

In [ ]:
if "pipeline_state" not in globals():
    raise RuntimeError("Execute a célula de fluxo principal (seção 5) antes desta visualização.")

# Bases já tratadas pelo pipeline
diario_plot = pipeline_state.diario_sem_outliers.copy().sort_values("date")
semanal_plot = pipeline_state.semanal.copy().sort_values("semana_inicio")

# Features auxiliares para visualização temporal longa
diario_plot["media_movel_30d"] = diario_plot["pedidos_dia"].rolling(30, min_periods=1).mean()
diario_plot["media_movel_90d"] = diario_plot["pedidos_dia"].rolling(90, min_periods=1).mean()

mensal_plot = (
    diario_plot.set_index("date")
    .resample("MS")
    .agg(pedidos_mes=("pedidos_dia", "sum"), itens_mes=("itens_dia", "sum"))
    .reset_index()
)
mensal_plot["ano"] = mensal_plot["date"].dt.year
mensal_plot["mes_num"] = mensal_plot["date"].dt.month

semanal_plot["ano"] = semanal_plot["semana_inicio"].dt.year
semanal_plot["semana_ano"] = semanal_plot["semana_inicio"].dt.isocalendar().week.astype(int)
heat_week = semanal_plot.pivot_table(index="semana_ano", columns="ano", values="pedidos_semana", aggfunc="mean")

corr_cols = [
    "pedidos_dia",
    "itens_dia",
    "linhas_itens",
    "is_holiday",
    "is_carnaval",
    "is_sao_joao",
    "is_summer",
]
corr_matrix = diario_plot[corr_cols].corr(numeric_only=True)

fig, axes = plt.subplots(2, 2, figsize=(18, 12), constrained_layout=True)

# 1) Correlação
sns.heatmap(
    corr_matrix,
    annot=True,
    fmt=".2f",
    cmap="RdBu_r",
    center=0,
    linewidths=0.5,
    ax=axes[0, 0],
)
axes[0, 0].set_title("Matriz de Correlação (base diária tratada)")

# 2) Série diária longa com tendências
axes[0, 1].plot(diario_plot["date"], diario_plot["pedidos_dia"], color="#5A7CA8", alpha=0.45, label="Pedidos diários")
axes[0, 1].plot(diario_plot["date"], diario_plot["media_movel_30d"], color="#E07A5F", linewidth=2.0, label="Média móvel 30d")
axes[0, 1].plot(diario_plot["date"], diario_plot["media_movel_90d"], color="#3D405B", linewidth=2.2, label="Média móvel 90d")
axes[0, 1].set_title("Série temporal diária longa (2023-2026)")
axes[0, 1].set_xlabel("Data")
axes[0, 1].set_ylabel("Pedidos")
axes[0, 1].legend()

# 3) Série mensal longa (agregada)
for ano, grp in mensal_plot.groupby("ano"):
    axes[1, 0].plot(grp["date"], grp["pedidos_mes"], marker="o", linewidth=1.8, label=str(ano))
axes[1, 0].set_title("Pedidos mensais por ano")
axes[1, 0].set_xlabel("Mês")
axes[1, 0].set_ylabel("Pedidos no mês")
axes[1, 0].legend(title="Ano", ncol=2)

# 4) Heatmap sazonal semanal por ano
sns.heatmap(
    heat_week,
    cmap="YlOrRd",
    linewidths=0.3,
    linecolor="white",
    ax=axes[1, 1],
)
axes[1, 1].set_title("Sazonalidade: pedidos por semana do ano")
axes[1, 1].set_xlabel("Ano")
axes[1, 1].set_ylabel("Semana do ano")

plt.show()

print("Correlação com pedidos_dia (ordem decrescente):")
display(corr_matrix["pedidos_dia"].sort_values(ascending=False).to_frame("corr_com_pedidos_dia"))

print("Amostra das séries longas mensais:")
display(mensal_plot.tail(12))

## Parte 3 — Comparação entre Modelos

Treinamento de múltiplos algoritmos distintos (regressão linear, Random Forest e Gradient Boosting) sobre o mesmo split temporal, com comparação por RMSE, MAE e R². Como o problema é de **regressão** (previsão de pedidos semanais), as visualizações de apoio são:
- gráfico de predito vs. real,
- distribuição dos resíduos,
- erro absoluto ao longo do tempo,
- ranking de RMSE por modelo.

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

NUM_FEATURES = [
    "itens_semana",
    "feriado_semana",
    "carnaval_semana",
    "sao_joao_semana",
    "verao_semana",
    "semana_ano",
    "trend",
    "lag_1",
    "rolling_4",
]
CAT_FEATURES = ["mes"]


def make_preprocessor() -> ColumnTransformer:
    return ColumnTransformer(
        transformers=[
            ("num", StandardScaler(), NUM_FEATURES),
            ("cat", OneHotEncoder(handle_unknown="ignore"), CAT_FEATURES),
        ]
    )


def split_train_test(weekly_df: pd.DataFrame, test_weeks: int) -> tuple[pd.DataFrame, pd.DataFrame]:
    split_idx = len(weekly_df) - test_weeks
    return weekly_df.iloc[:split_idx].copy(), weekly_df.iloc[split_idx:].copy()


def compare_models(weekly_df: pd.DataFrame, test_weeks: int) -> dict[str, dict[str, Any]]:
    train, test = split_train_test(weekly_df, test_weeks)
    X_train, y_train = train[NUM_FEATURES + CAT_FEATURES], train["pedidos_semana"]
    X_test, y_test = test[NUM_FEATURES + CAT_FEATURES], test["pedidos_semana"]

    candidates = {
        "linear_regression": LinearRegression(),
        "random_forest": RandomForestRegressor(n_estimators=300, random_state=SEED),
        "gradient_boosting": GradientBoostingRegressor(n_estimators=300, random_state=SEED),
    }

    results: dict[str, dict[str, Any]] = {}
    for name, estimator in candidates.items():
        pipe = Pipeline([("prep", make_preprocessor()), ("reg", estimator)])
        pipe.fit(X_train, y_train)
        pred = pipe.predict(X_test)
        results[name] = {
            "model": pipe,
            "pred": pred,
            "rmse": float(np.sqrt(mean_squared_error(y_test, pred))),
            "mae": float(mean_absolute_error(y_test, pred)),
            "r2": float(r2_score(y_test, pred)),
        }
    return {"results": results, "test": test, "y_test": y_test}


comparison = compare_models(pipeline_state.semanal, test_weeks=CONFIG["test_weeks"])
metrics_df = pd.DataFrame(
    {name: {k: v for k, v in info.items() if k in {"rmse", "mae", "r2"}} for name, info in comparison["results"].items()}
).T.sort_values("rmse")
print("Métricas comparativas no conjunto de teste:")
display(metrics_df)

best_name = metrics_df.index[0]
print(f"Melhor modelo por RMSE: {best_name}")

test_frame = comparison["test"][["semana_inicio", "pedidos_semana"]].copy()
for name, info in comparison["results"].items():
    test_frame[f"pred_{name}"] = info["pred"]

fig, axes = plt.subplots(2, 2, figsize=(16, 11), constrained_layout=True)

# (1) Predito vs Real
for name, info in comparison["results"].items():
    axes[0, 0].scatter(test_frame["pedidos_semana"], info["pred"], alpha=0.7, label=name)
lim_min = test_frame["pedidos_semana"].min()
lim_max = test_frame["pedidos_semana"].max()
axes[0, 0].plot([lim_min, lim_max], [lim_min, lim_max], color="black", linestyle="--", label="ideal")
axes[0, 0].set_title("Predito vs. Real")
axes[0, 0].set_xlabel("Pedidos reais")
axes[0, 0].set_ylabel("Pedidos previstos")
axes[0, 0].legend()

# (2) Distribuição de resíduos
for name, info in comparison["results"].items():
    residuals = test_frame["pedidos_semana"].values - info["pred"]
    sns.kdeplot(residuals, ax=axes[0, 1], label=name, fill=True, alpha=0.25)
axes[0, 1].axvline(0, color="black", linestyle="--")
axes[0, 1].set_title("Distribuição dos resíduos (real − previsto)")
axes[0, 1].set_xlabel("Resíduo")
axes[0, 1].legend()

# (3) Erro absoluto ao longo do tempo
for name, info in comparison["results"].items():
    abs_err = np.abs(test_frame["pedidos_semana"].values - info["pred"])
    axes[1, 0].plot(test_frame["semana_inicio"], abs_err, marker="o", label=name)
axes[1, 0].set_title("Erro absoluto por semana de teste")
axes[1, 0].set_xlabel("Semana")
axes[1, 0].set_ylabel("|real − previsto|")
axes[1, 0].legend()

# (4) Ranking RMSE
sns.barplot(
    x=metrics_df.index,
    y=metrics_df["rmse"].values,
    ax=axes[1, 1],
    palette="viridis",
)
axes[1, 1].set_title("RMSE por modelo (menor é melhor)")
axes[1, 1].set_ylabel("RMSE")
axes[1, 1].set_xlabel("")

plt.show()

## Parte 4 — Ajuste de Hiperparâmetros

Aplicação de **Grid Search** sobre o melhor modelo da Parte 3 utilizando `TimeSeriesSplit` (apropriado para séries temporais — preserva a ordem cronológica e evita vazamento de dados futuros). A entrega contém o grid de parâmetros avaliado, o melhor conjunto encontrado e o RMSE final no conjunto de teste reservado.

In [ ]:
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit


def tune_model(weekly_df: pd.DataFrame, model_name: str, test_weeks: int) -> dict[str, Any]:
    train, test = split_train_test(weekly_df, test_weeks)
    X_train, y_train = train[NUM_FEATURES + CAT_FEATURES], train["pedidos_semana"]
    X_test, y_test = test[NUM_FEATURES + CAT_FEATURES], test["pedidos_semana"]

    if model_name == "random_forest":
        estimator = RandomForestRegressor(random_state=SEED)
        param_grid = {
            "reg__n_estimators": [200, 400, 800],
            "reg__max_depth": [None, 6, 12],
            "reg__min_samples_split": [2, 5],
        }
    elif model_name == "gradient_boosting":
        estimator = GradientBoostingRegressor(random_state=SEED)
        param_grid = {
            "reg__n_estimators": [200, 400, 800],
            "reg__learning_rate": [0.03, 0.05, 0.1],
            "reg__max_depth": [2, 3, 4],
        }
    else:
        estimator = LinearRegression()
        param_grid = {"reg__fit_intercept": [True, False]}

    pipe = Pipeline([("prep", make_preprocessor()), ("reg", estimator)])
    tscv = TimeSeriesSplit(n_splits=5)

    search = GridSearchCV(
        pipe,
        param_grid=param_grid,
        scoring="neg_root_mean_squared_error",
        cv=tscv,
        n_jobs=-1,
        refit=True,
    )
    search.fit(X_train, y_train)

    pred = search.best_estimator_.predict(X_test)
    rmse_test = float(np.sqrt(mean_squared_error(y_test, pred)))

    return {
        "best_params": search.best_params_,
        "best_cv_rmse": float(-search.best_score_),
        "test_rmse": rmse_test,
        "test_pred": pred,
        "estimator": search.best_estimator_,
        "cv_results": pd.DataFrame(search.cv_results_),
    }


tuning = tune_model(pipeline_state.semanal, model_name=best_name, test_weeks=CONFIG["test_weeks"])
print(f"Modelo afinado: {best_name}")
print("Melhor combinação de hiperparâmetros:")
for k, v in tuning["best_params"].items():
    print(f"  - {k}: {v}")
print(f"RMSE médio CV (treino): {tuning['best_cv_rmse']:.3f}")
print(f"RMSE no conjunto de teste reservado: {tuning['test_rmse']:.3f}")

cv_summary = (
    tuning["cv_results"][["mean_test_score", "std_test_score", "params"]]
    .assign(mean_test_rmse=lambda d: -d["mean_test_score"])
    .sort_values("mean_test_rmse")
    .head(10)
    .reset_index(drop=True)
)
print("Top 10 combinações testadas (menor RMSE):")
display(cv_summary[["mean_test_rmse", "std_test_score", "params"]])

## Parte 5 — Validação Cruzada e Análise de Variância

Avaliação do modelo afinado através de **validação cruzada temporal** (`TimeSeriesSplit`), com:
- distribuição do RMSE entre folds (variância → estabilidade do modelo),
- curva de aprendizado em função do tamanho do conjunto de treino (diagnóstico de overfitting/underfitting),
- diferença treino × validação para detectar gap.

In [ ]:
from sklearn.model_selection import cross_val_score, learning_curve


def cross_validate_model(estimator: Pipeline, weekly_df: pd.DataFrame, n_splits: int = 5) -> dict[str, Any]:
    X = weekly_df[NUM_FEATURES + CAT_FEATURES]
    y = weekly_df["pedidos_semana"]
    tscv = TimeSeriesSplit(n_splits=n_splits)

    fold_scores = -cross_val_score(
        estimator,
        X,
        y,
        cv=tscv,
        scoring="neg_root_mean_squared_error",
        n_jobs=-1,
    )

    train_sizes, train_scores, val_scores = learning_curve(
        estimator,
        X,
        y,
        cv=tscv,
        scoring="neg_root_mean_squared_error",
        train_sizes=np.linspace(0.3, 1.0, 6),
        n_jobs=-1,
        shuffle=False,
    )

    return {
        "fold_rmse": fold_scores,
        "train_sizes": train_sizes,
        "train_rmse": -train_scores,
        "val_rmse": -val_scores,
    }


cv_report = cross_validate_model(tuning["estimator"], pipeline_state.semanal, n_splits=5)

print("RMSE por fold (validação temporal):")
for i, score in enumerate(cv_report["fold_rmse"], start=1):
    print(f"  Fold {i}: {score:.3f}")
print(f"Média: {cv_report['fold_rmse'].mean():.3f}")
print(f"Desvio-padrão: {cv_report['fold_rmse'].std():.3f}")
print(f"Coef. variação: {cv_report['fold_rmse'].std() / cv_report['fold_rmse'].mean():.2%}")

train_mean = cv_report["train_rmse"].mean(axis=1)
val_mean = cv_report["val_rmse"].mean(axis=1)
gap = val_mean - train_mean

if gap[-1] > train_mean[-1] * 0.5:
    diagnostico = "indício de overfitting (validação muito acima do treino)"
elif train_mean[-1] > val_mean[-1] * 1.2:
    diagnostico = "possível underfitting"
else:
    diagnostico = "treino e validação próximos — modelo estável"
print(f"Diagnóstico de ajuste: {diagnostico}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)

axes[0].bar(range(1, len(cv_report["fold_rmse"]) + 1), cv_report["fold_rmse"], color="#5A7CA8")
axes[0].axhline(cv_report["fold_rmse"].mean(), color="red", linestyle="--", label="média")
axes[0].set_title("RMSE por fold (TimeSeriesSplit)")
axes[0].set_xlabel("Fold")
axes[0].set_ylabel("RMSE")
axes[0].legend()

axes[1].plot(cv_report["train_sizes"], train_mean, marker="o", label="treino", color="#3D405B")
axes[1].plot(cv_report["train_sizes"], val_mean, marker="o", label="validação", color="#E07A5F")
axes[1].fill_between(
    cv_report["train_sizes"],
    train_mean - cv_report["train_rmse"].std(axis=1),
    train_mean + cv_report["train_rmse"].std(axis=1),
    alpha=0.15,
    color="#3D405B",
)
axes[1].fill_between(
    cv_report["train_sizes"],
    val_mean - cv_report["val_rmse"].std(axis=1),
    val_mean + cv_report["val_rmse"].std(axis=1),
    alpha=0.15,
    color="#E07A5F",
)
axes[1].set_title("Curva de aprendizado")
axes[1].set_xlabel("Nº de amostras de treino")
axes[1].set_ylabel("RMSE")
axes[1].legend()

plt.show()

## Parte 6 — Função de Inferência

Estrutura mínima de integração do modelo treinado: uma função `predict_weekly_orders` que recebe o histórico semanal e retorna a previsão para a próxima semana, junto com a montagem das features necessárias (lag, média móvel, flags de evento). Esta função é o ponto de entrada para um futuro endpoint do produto.

In [ ]:
def build_inference_row(history: pd.DataFrame, target_week: pd.Timestamp, event_flags: dict[str, int]) -> pd.DataFrame:
    history = history.sort_values("semana_inicio").reset_index(drop=True)
    if history.empty:
        raise ValueError("Histórico semanal vazio.")

    last = history.iloc[-1]
    rolling_4 = float(history["pedidos_semana"].tail(4).mean())
    trend_value = float(last["trend"] + 1)
    itens_estimado = float(history["itens_semana"].tail(4).mean())

    row = {
        "semana_inicio": target_week,
        "itens_semana": itens_estimado,
        "feriado_semana": int(event_flags.get("feriado", 0)),
        "carnaval_semana": int(event_flags.get("carnaval", 0)),
        "sao_joao_semana": int(event_flags.get("sao_joao", 0)),
        "verao_semana": int(event_flags.get("verao", 0)),
        "ano": target_week.year,
        "mes": str(target_week.month),
        "semana_ano": int(target_week.isocalendar().week),
        "trend": trend_value,
        "lag_1": float(last["pedidos_semana"]),
        "rolling_4": rolling_4,
    }
    return pd.DataFrame([row])


def predict_weekly_orders(
    estimator: Pipeline,
    history: pd.DataFrame,
    target_week: pd.Timestamp | None = None,
    event_flags: dict[str, int] | None = None,
) -> dict[str, Any]:
    if target_week is None:
        last_week = history["semana_inicio"].max()
        target_week = last_week + pd.Timedelta(days=7)
    if event_flags is None:
        event_flags = {"feriado": 0, "carnaval": 0, "sao_joao": 0, "verao": 0}

    inference_row = build_inference_row(history, target_week, event_flags)
    X_inf = inference_row[NUM_FEATURES + CAT_FEATURES]
    pred = float(estimator.predict(X_inf)[0])

    return {
        "semana_inicio": target_week,
        "pedidos_previstos": pred,
        "features": inference_row.iloc[0].to_dict(),
    }


# Demonstração: prevê a próxima semana após o último ponto disponível.
final_estimator = tuning["estimator"]
prev_resultado = predict_weekly_orders(final_estimator, pipeline_state.semanal)
print("Previsão para a próxima semana:")
print(f"  Semana de início: {prev_resultado['semana_inicio'].date()}")
print(f"  Pedidos previstos: {prev_resultado['pedidos_previstos']:.1f}")

# Cenário com evento (feriado): demonstra o uso pelo time de produto.
prev_feriado = predict_weekly_orders(
    final_estimator,
    pipeline_state.semanal,
    event_flags={"feriado": 1, "carnaval": 0, "sao_joao": 0, "verao": 0},
)
print("Previsão alternativa (semana com feriado):")
print(f"  Pedidos previstos: {prev_feriado['pedidos_previstos']:.1f}")

## Parte 7 — Pipeline Completo com Modelo Final

Execução end-to-end com os dados atualizados, utilizando o **modelo final ajustado**:
1. carga e limpeza,
2. construção da base semanal,
3. predição com o estimador afinado,
4. métricas finais (RMSE, MAE, R²) e evidências de estabilidade do modelo.

In [ ]:
def run_full_pipeline(config: dict[str, Any], final_model: Pipeline) -> dict[str, Any]:
    raw = load_tables(config)
    clean = normalize_tables(raw)
    daily = create_daily_frame(clean)
    daily_clean, _, _ = remove_outliers_iqr_with_exceptions(
        daily,
        target_col="pedidos_dia",
        event_cols=config["event_exception_cols"],
        multiplier=config["iqr_multiplier"],
    )
    weekly = build_weekly_dataset(daily_clean)

    train, test = split_train_test(weekly, config["test_weeks"])
    X_test, y_test = test[NUM_FEATURES + CAT_FEATURES], test["pedidos_semana"]
    pred = final_model.predict(X_test)

    metrics = {
        "rmse": float(np.sqrt(mean_squared_error(y_test, pred))),
        "mae": float(mean_absolute_error(y_test, pred)),
        "r2": float(r2_score(y_test, pred)),
        "mape": float(np.mean(np.abs((y_test.values - pred) / y_test.values)) * 100),
    }

    cv_scores = -cross_val_score(
        final_model,
        weekly[NUM_FEATURES + CAT_FEATURES],
        weekly["pedidos_semana"],
        cv=TimeSeriesSplit(n_splits=5),
        scoring="neg_root_mean_squared_error",
        n_jobs=-1,
    )
    stability = {
        "rmse_cv_mean": float(cv_scores.mean()),
        "rmse_cv_std": float(cv_scores.std()),
        "rmse_cv_cv": float(cv_scores.std() / cv_scores.mean()),
    }

    return {
        "weekly": weekly,
        "test": test,
        "y_test": y_test,
        "predictions": pred,
        "metrics": metrics,
        "stability": stability,
    }


final_run = run_full_pipeline(CONFIG, final_estimator)

print("Métricas finais (conjunto de teste reservado):")
for metric, value in final_run["metrics"].items():
    print(f"  - {metric.upper()}: {value:.3f}")

print("Estabilidade do modelo (validação cruzada temporal):")
for k, v in final_run["stability"].items():
    print(f"  - {k}: {v:.3f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)

axes[0].plot(final_run["test"]["semana_inicio"], final_run["y_test"].values, marker="o", label="Real")
axes[0].plot(final_run["test"]["semana_inicio"], final_run["predictions"], marker="o", label="Previsto (final)")
axes[0].set_title("Pipeline final - previsão semanal")
axes[0].set_xlabel("Semana")
axes[0].set_ylabel("Pedidos por semana")
axes[0].legend()

residuos_finais = final_run["y_test"].values - final_run["predictions"]
axes[1].axhline(0, color="black", linestyle="--")
axes[1].scatter(final_run["predictions"], residuos_finais, alpha=0.7, color="#E07A5F")
axes[1].set_title("Resíduos do modelo final")
axes[1].set_xlabel("Pedidos previstos")
axes[1].set_ylabel("Resíduo (real − previsto)")

plt.show()

# Persiste o modelo final no estado do pipeline para reuso.
pipeline_state.model_artifacts = ModelArtifacts(
    rmse={"final": final_run["metrics"]["rmse"], **{k: v["rmse"] for k, v in comparison["results"].items()}},
    test_frame=final_run["test"][["semana_inicio", "pedidos_semana"]].assign(pred_final=final_run["predictions"]),
    model=final_estimator,
)
print("Pipeline completo executado com modelo final ajustado.")

## Conclusão

Pipeline modular e reprodutível cobrindo as sete entregas do roteiro CC:

| Parte | Entrega | Onde está |
|-------|---------|-----------|
| 1 | Carga, EDA inicial, limpeza, normalização e tratamento de outliers | Seções 1–5 |
| 2 | Modelo baseline (regressão linear) com métricas em validação | Seções 4–5 |
| 3 | Comparação de algoritmos distintos (LR, Random Forest, Gradient Boosting) com visualizações de erro | Parte 3 |
| 4 | Ajuste de hiperparâmetros via Grid Search com `TimeSeriesSplit` | Parte 4 |
| 5 | Validação cruzada temporal, curva de aprendizado e diagnóstico de over/underfitting | Parte 5 |
| 6 | Função de inferência `predict_weekly_orders` para integração no produto | Parte 6 |
| 7 | Pipeline completo end-to-end com modelo final, métricas e evidência de estabilidade | Parte 7 |

Próximos passos recomendados:
1. Substituir o estimador de `itens_semana` na função de inferência por um modelo dedicado ou por previsão explícita do usuário do produto.
2. Avaliar features de estoque (entradas, validade) como variáveis adicionais.
3. Considerar modelos sequenciais (SARIMAX, Prophet) como benchmark para a sazonalidade anual.